# Future weather attributes — one-year LSTM forecast

Run from top to bottom. Set the dataset/model paths and forecast dates in the settings cell. The existing LSTM and scalers are reused when saved artifacts exist; training runs only when needed or explicitly requested.

```text
Future Weather Attributes/
├── Data/
├── Forecast/                 # future_daily_weather_attributes.csv
├── experiment_logs/          # metrics, historical predictions and plots
├── trained_models/           # model.keras, scalers.joblib and config.json
├── Future_Weather_Attributes_LSTM.ipynb
└── README.md
```

The default forecast is **2026-02-01 through 2027-01-31**: one complete year from the requested start. Set `FORECAST_END="2026-12-31"` for just the remainder of calendar 2026. Forecasts are recursive: every generated day supplies weather history for the next day. Town coordinates remain fixed weather-station metadata.

Metrics compare forecasts against historical observations. January one-step scores and January recursive scores are different evaluations; neither establishes accuracy of the unobserved future year. The old saved seven-day outputs have been cleared.

## 1. Settings

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "Future Weather Attributes" / "Future_Weather_Attributes_LSTM.ipynb").is_file():
    ROOT = ROOT / "Future Weather Attributes"
EXPERIMENT_ID = "baseline_v1"
DATA_PATH = ROOT / "Data/LSTM_processed/japan_daily_lstm_processed.csv"
MODEL_DIR = ROOT / "trained_models/future_weather" / EXPERIMENT_ID
FORECAST_DIR = ROOT / "Forecast" / EXPERIMENT_ID
LOG_DIR = ROOT / "experiment_logs" / EXPERIMENT_ID

LOOKBACK = 30
EPOCHS = 30
BATCH_SIZE = 128
MAX_TRAIN_WINDOWS = 160_000
SEED = 42
TRAIN_IF_MISSING = True
FORCE_RETRAIN = False

TRAIN_START = "2021-01-01"
TRAIN_END = "2025-09-30"
VAL_START = "2025-10-01"
VAL_END = "2025-12-31"
TEST_START = "2026-01-01"
TEST_END = "2026-01-31"
RUN_EVALUATION = True

FORECAST_START = "2026-02-01"  # None starts the day after the latest selected history
FORECAST_END = None            # None = one calendar year from start, inclusive
FORECAST_PREFECTURE = None     # None = all prefectures
FORECAST_TOWN = None           # None = all towns in the selected prefecture(s)
RUN_FORECAST = True
FORECAST_BATCH_SIZE = 256
PROGRESS_EVERY = 30


## 2. Imports, folders and old-output migration

In [ ]:
# Install once if needed:
# %pip install numpy pandas matplotlib scikit-learn tensorflow joblib
%matplotlib inline
import json, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from IPython.display import display

assert LOOKBACK > 0 and EPOCHS > 0 and BATCH_SIZE > 0
assert MAX_TRAIN_WINDOWS is None or MAX_TRAIN_WINDOWS > 0
assert FORECAST_BATCH_SIZE > 0
for folder in [MODEL_DIR, FORECAST_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Preserve old contents while replacing Forecast_outputs with the two new folders.
legacy = ROOT / "Forecast_outputs"
if legacy.exists():
    for source in list(legacy.rglob("*")):
        if not source.is_file():
            continue
        relative = source.relative_to(legacy)
        target_root = ROOT / "Forecast" if source.name == "future_daily_weather_attributes.csv" else ROOT / "experiment_logs"
        target = target_root / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            number = 1
            while target.with_name(f"{target.stem}_legacy{number}{target.suffix}").exists():
                number += 1
            target = target.with_name(f"{target.stem}_legacy{number}{target.suffix}")
        shutil.move(str(source), str(target))
        print("Preserved old output:", target)
    for directory in sorted(legacy.rglob("*"), key=lambda p: len(p.parts), reverse=True):
        if directory.is_dir() and not any(directory.iterdir()):
            directory.rmdir()
    if not any(legacy.iterdir()):
        legacy.rmdir()

# Move existing metric/plot files out of trained_models; keep only model artifacts there.
for source in list(MODEL_DIR.glob("*.csv")) + list(MODEL_DIR.glob("*.png")):
    target = LOG_DIR / source.name
    if not target.exists():
        shutil.move(str(source), str(target))

if not DATA_PATH.is_file():
    raise FileNotFoundError("Set DATA_PATH to your historical japan_daily_lstm_processed.csv; the seven-day forecast CSV is not the full history.")
print("Historical data:", DATA_PATH)
print("Future CSV:", FORECAST_DIR / "future_daily_weather_attributes.csv")
print("Metrics and plots:", LOG_DIR)


## 3. Load the exact weather feature contract
Existing saved feature order is reused, including circular wind direction. Extra metadata or `pressure_change` columns are not accidentally added to the model inputs.

In [ ]:
DEFAULT_WEATHER = [
    "temperature_2m_max",
    "temperature_2m_min",
    "wind_speed_10m_max",
    "wind_gusts_10m_max",
    "snowfall_sum",
    "rain_sum",
    "precipitation_hours",
    "precipitation_sum",
    "shortwave_radiation_sum",
    "sunshine_duration",
    "daylight_duration",
    "dew_point_2m_mean",
    "dew_point_2m_max",
    "surface_pressure_mean",
    "surface_pressure_max",
    "surface_pressure_min",
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "relative_humidity_2m_max",
    "relative_humidity_2m_min",
    "dew_point_2m_min",
    "cloud_cover_mean",
    "cloud_cover_max",
    "cloud_cover_min",
    "pressure_msl_mean",
    "pressure_msl_max",
    "pressure_msl_min",
    "wind_speed_10m_mean",
    "wind_speed_10m_min",
    "wind_gusts_10m_mean",
    "wind_gusts_10m_min",
    "river_discharge",
    "river_discharge_p75",
    "river_discharge_mean",
    "river_discharge_median",
    "river_discharge_max",
    "wind_direction_10m_dominant"
]
ANGLE = "wind_direction_10m_dominant"
ARTIFACTS = [MODEL_DIR / name for name in ["model.keras", "scalers.joblib", "config.json"]]
REUSE_MODEL = all(p.is_file() for p in ARTIFACTS) and not FORCE_RETRAIN
if not REUSE_MODEL and any(p.is_file() for p in ARTIFACTS) and not FORCE_RETRAIN:
    raise FileNotFoundError("Incomplete model artifacts. Restore all three files, choose a new EXPERIMENT_ID, or set FORCE_RETRAIN=True.")
if REUSE_MODEL:
    config = json.loads((MODEL_DIR / "config.json").read_text(encoding="utf-8"))
    features = config["features"]
    LOOKBACK = int(config["lookback"])
else:
    features = [c for c in DEFAULT_WEATHER if c != ANGLE] + ["wind_direction_sin", "wind_direction_cos"]
if not features or len(features) != len(set(features)):
    raise ValueError("Saved feature order must be nonempty and unique.")

def load_data(path, ordered_features):
    data = pd.read_csv(path, low_memory=False, encoding="utf-8-sig")
    required = {"prefecture", "town", "date", "latitude", "longitude"}
    if not required.issubset(data):
        raise ValueError(f"Historical data requires {sorted(required)}.")
    if "elevation" not in data and "openmeteo_elevation" in data:
        data["elevation"] = data["openmeteo_elevation"]
    if "elevation" not in data:
        raise ValueError("Historical stations require elevation or openmeteo_elevation.")
    data["date"] = pd.to_datetime(data.date, errors="raise").dt.normalize()
    if data[["prefecture", "town", "date"]].isna().any().any():
        raise ValueError("Missing town/date keys.")
    if data.duplicated(["prefecture", "town", "date"]).any():
        raise ValueError("Duplicate town/date records.")
    if ANGLE in data:
        direction = np.deg2rad(pd.to_numeric(data[ANGLE], errors="raise"))
        data["wind_direction_sin"], data["wind_direction_cos"] = np.sin(direction), np.cos(direction)
    if ANGLE not in data and {"wind_direction_sin", "wind_direction_cos"}.issubset(data):
        data[ANGLE] = np.degrees(np.arctan2(data.wind_direction_sin, data.wind_direction_cos)) % 360
    missing = set(ordered_features) - set(data)
    if missing:
        raise ValueError(f"Historical weather lacks saved model features: {sorted(missing)}")
    for name in list(ordered_features) + ["latitude", "longitude", "elevation"]:
        data[name] = pd.to_numeric(data[name], errors="raise")
    if not np.isfinite(data[list(ordered_features) + ["latitude", "longitude", "elevation"]].to_numpy()).all():
        raise ValueError("Nonfinite weather or station metadata. Use the existing preprocessing to repair historical data.")
    return data.sort_values(["prefecture", "town", "date"]).reset_index(drop=True)

df = load_data(DATA_PATH, features)
print(f"{len(df):,} historical rows | {df.groupby(['prefecture','town']).ngroups} towns | {len(features)} model features")
print("Available dates:", df.date.min().date(), "through", df.date.max().date())
print("Use saved model:", REUSE_MODEL, "| Lookback:", LOOKBACK)
display(df.head(3))


## 4. Weather forecast helpers

In [ ]:
"""Batched daily recursive weather forecasting; usable without TensorFlow imports."""

import numpy as np
import pandas as pd

ANGLE = "wind_direction_10m_dominant"
DIRECTION_SIN = "wind_direction_sin"
DIRECTION_COS = "wind_direction_cos"
NONNEG = (
    "rain_", "precipitation_", "snowfall_", "sunshine_duration",
    "daylight_duration", "shortwave_radiation_sum", "wind_speed_",
    "wind_gusts_", "river_discharge",
)
PERCENT = ("relative_humidity_", "cloud_cover_")


def calendar(dates):
    dates = pd.DatetimeIndex(dates)
    angle = 2 * np.pi * (dates.dayofyear.to_numpy() - 1) / 365.25
    return np.column_stack((np.sin(angle), np.cos(angle))).astype("float32")


def _positive_integer(value, name):
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value <= 0:
        raise ValueError(f"{name} must be a positive integer")
    return int(value)


def _day_timestamp(value, name):
    result = pd.Timestamp(value)
    if pd.isna(result) or result.tzinfo is not None or result != result.normalize():
        raise ValueError(f"{name} must be a timezone-free daily date")
    return result


def _as_finite_matrix(values, expected_shape, name):
    array = np.asarray(values, dtype=np.float32)
    if array.shape != expected_shape:
        raise ValueError(f"{name} shape {array.shape}, expected {expected_shape}")
    if not np.isfinite(array).all():
        raise ValueError(f"{name} contains nonfinite values")
    return array


def _bound_weather(raw, features):
    """Apply the same physical output constraints before output and feedback."""
    bounded = raw.copy()
    positions = {name: index for index, name in enumerate(features)}
    for name, column in positions.items():
        if name.startswith(NONNEG):
            bounded[:, column] = np.maximum(0, bounded[:, column])
        if name.startswith(PERCENT):
            bounded[:, column] = np.clip(bounded[:, column], 0, 100)
        if name == "precipitation_hours":
            bounded[:, column] = np.clip(bounded[:, column], 0, 24)
        if name in ("sunshine_duration", "daylight_duration"):
            bounded[:, column] = np.clip(bounded[:, column], 0, 86400)
    # Project any available min/mean/max family onto increasing order.
    families = {}
    for name in features:
        for suffix in ("_min", "_mean", "_max"):
            if name.endswith(suffix):
                families.setdefault(name[:-len(suffix)], {})[suffix] = positions[name]
    for family in families.values():
        ordered = [family[s] for s in ("_min", "_mean", "_max") if s in family]
        if len(ordered) > 1:
            bounded[:, ordered] = np.sort(bounded[:, ordered], axis=1)
    if "sunshine_duration" in positions and "daylight_duration" in positions:
        bounded[:, positions["sunshine_duration"]] = np.minimum(
            bounded[:, positions["sunshine_duration"]], bounded[:, positions["daylight_duration"]]
        )
    if DIRECTION_SIN in positions and DIRECTION_COS in positions:
        angle = np.arctan2(bounded[:, positions[DIRECTION_SIN]], bounded[:, positions[DIRECTION_COS]])
        bounded[:, positions[DIRECTION_SIN]] = np.sin(angle)
        bounded[:, positions[DIRECTION_COS]] = np.cos(angle)
    if ANGLE in positions:
        bounded[:, positions[ANGLE]] %= 360
    return bounded


def recursive_forecast(
    history_df, model, scaler, loc_scaler, features, lookback,
    start, end, prefecture=None, town=None, batch_size=128, progress_every=30,
):
    """Forecast every selected town/day in the inclusive start..end interval.

    Only rows strictly before start may initialize the model. Every selected
    station must have a complete consecutive lookback ending on start - 1 day.
    model must be callable as model([past, target_calendar], training=False).
    Observations on/after start are never read into forecast state.
    """
    lookback = _positive_integer(lookback, "lookback")
    batch_size = _positive_integer(batch_size, "batch_size")
    if progress_every == 0 and isinstance(progress_every, (int, np.integer)) and not isinstance(progress_every, (bool, np.bool_)):
        progress_every = None
    if progress_every is not None:
        progress_every = _positive_integer(progress_every, "progress_every")
    start = _day_timestamp(start, "start")
    end = _day_timestamp(end, "end")
    if end < start:
        raise ValueError("end must be on or after start")
    features = list(features)
    if not features or any(not isinstance(name, str) for name in features) or len(set(features)) != len(features):
        raise ValueError("features must be nonempty unique names, in saved scaler order")
    metadata = ["prefecture", "town", "date", "latitude", "longitude", "elevation"]
    if set(features).intersection(metadata + ["year", "month", "day"]):
        raise ValueError("Saved weather target features cannot contain station/calendar metadata")
    absent = set(metadata) - set(history_df.columns)
    if absent:
        raise ValueError(f"History lacks required station metadata: {sorted(absent)}")
    df = history_df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    if df["date"].isna().any() or df["date"].dt.tz is not None:
        raise ValueError("History dates must be finite timezone-free daily dates")
    if not df["date"].eq(df["date"].dt.normalize()).all():
        raise ValueError("History contains timestamps that are not daily midnight dates")
    if prefecture is not None:
        df = df.loc[df["prefecture"] == prefecture].copy()
    if town is not None:
        df = df.loc[df["town"] == town].copy()
    if df.empty:
        raise ValueError("No selected towns in history input")
    if df[["prefecture", "town"]].isna().any().any():
        raise ValueError("Station prefecture/town keys cannot be missing")
    selected_keys = set(df[["prefecture", "town"]].itertuples(index=False, name=None))
    # Inspect only keys/dates above; future weather/metadata never enter seed state.
    df = df.loc[df["date"] < start].copy()
    historical_keys = set(df[["prefecture", "town"]].itertuples(index=False, name=None))
    missing_seeds = selected_keys - historical_keys
    if missing_seeds:
        labels = ", ".join(f"{pref}/{town_name}" for pref, town_name in sorted(missing_seeds))
        raise ValueError(f"Missing historical seed for selected towns: {labels}; no observations before forecast start")
    if df.duplicated(["prefecture", "town", "date"]).any():
        raise ValueError("Duplicate historical station/date records")
    direction_features = {DIRECTION_SIN, DIRECTION_COS}.intersection(features)
    if direction_features and len(direction_features) != 2:
        raise ValueError("Wind direction requires both saved sine and cosine features")
    if direction_features and ANGLE in df.columns:
        # Encode from the raw angle rather than trusting potentially stale helper columns.
        degrees = pd.to_numeric(df[ANGLE], errors="raise")
        radians = np.deg2rad(degrees)
        df[DIRECTION_SIN] = np.sin(radians)
        df[DIRECTION_COS] = np.cos(radians)
    absent = set(features) - set(df.columns)
    if absent:
        raise ValueError(f"History lacks saved model features: {sorted(absent)}")
    for name in features + ["latitude", "longitude", "elevation"]:
        df[name] = pd.to_numeric(df[name], errors="raise")

    histories, calendars, locations, stations, previous_pressures = [], [], [], [], []
    for (pref, town_name), g in df.groupby(["prefecture", "town"], sort=False):
        g = g.sort_values("date")
        label = f"{pref}/{town_name}"
        if len(g) < lookback:
            raise ValueError(f"{label}: need {lookback} historical seed days, found {len(g)}")
        if g["date"].iloc[-1] != start - pd.Timedelta(days=1):
            raise ValueError(f"{label}: history must end the day before forecast start")
        seed = g.tail(lookback)
        expected = pd.date_range(start - pd.Timedelta(days=lookback), periods=lookback, freq="D")
        if not np.array_equal(seed["date"].to_numpy(), expected.to_numpy()):
            raise ValueError(f"{label}: nonconsecutive or incomplete historical seed days")
        metadata_values = g[["latitude", "longitude", "elevation"]].to_numpy(dtype="float64")
        if not np.isfinite(metadata_values).all():
            raise ValueError(f"{label}: nonfinite station metadata")
        if not np.allclose(metadata_values, metadata_values[-1], rtol=0, atol=1e-6):
            raise ValueError(f"{label}: station coordinates/elevation must remain static")
        lat, lon, elevation = metadata_values[-1]
        if not -90 <= lat <= 90 or not -180 <= lon <= 180:
            raise ValueError(f"{label}: invalid latitude/longitude")
        seed_raw = _as_finite_matrix(seed[features].to_numpy(), (lookback, len(features)), f"{label} weather seed")
        histories.append(_as_finite_matrix(scaler.transform(seed_raw), seed_raw.shape, f"{label} scaled weather seed"))
        calendars.append(calendar(seed["date"]))
        locations.append(metadata_values[-1])
        stations.append((pref, town_name, float(lat), float(lon), float(elevation)))
        if "surface_pressure_mean" in features:
            previous_pressures.append(float(seed_raw[-1, features.index("surface_pressure_mean")]))
    station_count = len(stations)
    history = np.stack(histories)
    histcal = np.stack(calendars)
    locations_scaled = _as_finite_matrix(
        loc_scaler.transform(np.asarray(locations)), (station_count, 3), "Scaled station metadata"
    )
    loc_windows = np.broadcast_to(locations_scaled[:, None, :], (station_count, lookback, 3))
    days = pd.date_range(start, end, freq="D")
    feature_positions = {name: i for i, name in enumerate(features)}
    previous_pressures = np.asarray(previous_pressures, dtype="float32")
    frames = []
    for step, day in enumerate(days, start=1):
        targetcal = calendar([day])
        targetcal_batch = np.broadcast_to(targetcal, (station_count, 2)).copy()
        scaled_predictions = []
        for offset in range(0, station_count, batch_size):
            stop = min(offset + batch_size, station_count)
            past = np.concatenate([history[offset:stop], histcal[offset:stop], loc_windows[offset:stop]], axis=2)
            output = model([past, targetcal_batch[offset:stop]], training=False)
            scaled_predictions.append(_as_finite_matrix(output, (stop-offset, len(features)), "Model prediction"))
        scaled_prediction = np.concatenate(scaled_predictions, axis=0)
        physical = _as_finite_matrix(scaler.inverse_transform(scaled_prediction), scaled_prediction.shape, "Physical prediction")
        bounded = _bound_weather(physical, features)
        delta = None
        if "surface_pressure_mean" in feature_positions:
            pressures = bounded[:, feature_positions["surface_pressure_mean"]]
            delta = pressures - previous_pressures
            previous_pressures = pressures.copy()
            if "pressure_change" in feature_positions:
                bounded[:, feature_positions["pressure_change"]] = delta
        records = []
        for index, (pref, town_name, lat, lon, elevation) in enumerate(stations):
            record = {
                "prefecture": pref, "town": town_name, "date": day.strftime("%Y-%m-%d"),
                "latitude": lat, "longitude": lon, "elevation": elevation,
                "year": day.year, "month": day.month, "day": day.day,
            }
            record.update({name: float(bounded[index, feature_positions[name]]) for name in features if name not in {DIRECTION_SIN, DIRECTION_COS}})
            if direction_features:
                sine = bounded[index, feature_positions[DIRECTION_SIN]]
                cosine = bounded[index, feature_positions[DIRECTION_COS]]
                record[ANGLE] = float(np.degrees(np.arctan2(sine, cosine)) % 360)
            if delta is not None:
                record["pressure_change"] = float(delta[index])
            records.append(record)
        frames.append(pd.DataFrame.from_records(records))
        # Feed the bounded physical predictions through the ORIGINAL saved scaler order.
        next_scaled = _as_finite_matrix(scaler.transform(bounded), bounded.shape, "Bounded scaled feedback")
        history = np.concatenate([history[:, 1:, :], next_scaled[:, None, :]], axis=1)
        histcal = np.concatenate([histcal[:, 1:, :], targetcal_batch[:, None, :]], axis=1)
        if progress_every and (step % progress_every == 0 or step == len(days)):
            print(f"Forecasted {step}/{len(days)} days for {station_count} towns; through {day.date()}", flush=True)
    result = pd.concat(frames, ignore_index=True)
    expected_rows = station_count * len(days)
    if len(result) != expected_rows or result.duplicated(["prefecture", "town", "date"]).any():
        raise AssertionError("Forecast output does not have exactly one row per selected town/day")
    counts = result.groupby(["prefecture", "town"], sort=False)["date"].nunique()
    if not counts.eq(len(days)).all():
        raise AssertionError("Forecast date coverage differs between towns")
    numeric_columns = result.select_dtypes(include=[np.number]).columns
    if not np.isfinite(result[numeric_columns].to_numpy(dtype="float64")).all():
        raise ValueError("Forecast output contains nonfinite numeric values")
    result.attrs["forecast_start"] = str(start.date())
    result.attrs["forecast_end"] = str(end.date())
    result.attrs["forecast_days"] = len(days)
    result.attrs["town_count"] = station_count
    return result


## 5. Prepare chronological training and test windows
Training ends before validation; heldout target dates are after both. Static locations are scaled once per town rather than inside every training example.

In [ ]:
def build_groups(data, feature_scaler, location_scaler):
    result = []
    for (prefecture, town), part in data.groupby(["prefecture", "town"], sort=False):
        dates = pd.DatetimeIndex(part.date)
        if not np.all(np.diff(dates.values).astype("timedelta64[D]").astype(int) == 1):
            raise ValueError(f"Nonconsecutive historical dates: {prefecture}/{town}.")
        locations = part[["latitude", "longitude", "elevation"]].to_numpy(dtype="float32")
        if not np.allclose(locations, locations[0], atol=1e-5, rtol=0):
            raise ValueError(f"Station coordinates changed: {prefecture}/{town}.")
        values = feature_scaler.transform(part[features].to_numpy(dtype="float32")).astype("float32")
        location = location_scaler.transform(locations[[0]])[0].astype("float32")
        result.append(dict(prefecture=prefecture, town=town, dates=dates, values=values,
                           calendar=calendar(dates), location=location))
    return result

def build_index(groups, first, last):
    pairs = []
    for group_id, group in enumerate(groups):
        targets = np.flatnonzero((group["dates"] >= pd.Timestamp(first)) & (group["dates"] <= pd.Timestamp(last)))
        pairs.extend((group_id, int(t)) for t in targets if t >= LOOKBACK)
    return np.asarray(pairs, dtype="int32").reshape(-1, 2)

def input_batch(pairs, groups):
    xx, target_calendar, yy = [], [], []
    for group_id, t in pairs:
        group = groups[int(group_id)]
        xx.append(np.concatenate([group["values"][t-LOOKBACK:t], group["calendar"][t-LOOKBACK:t],
                    np.broadcast_to(group["location"], (LOOKBACK, 3))], axis=1))
        target_calendar.append(group["calendar"][t])
        yy.append(group["values"][t])
    return np.stack(xx), np.stack(target_calendar), np.stack(yy)

if REUSE_MODEL:
    scales = joblib.load(MODEL_DIR / "scalers.joblib")
    scaler, loc_scaler = scales["scaler"], scales["loc_scaler"]
else:
    train_rows = df.date.between(TRAIN_START, TRAIN_END)
    if not train_rows.any():
        raise ValueError("No rows in the training period.")
    scaler = StandardScaler().fit(df.loc[train_rows, features].to_numpy(dtype="float32"))
    loc_scaler = StandardScaler().fit(df.loc[train_rows, ["latitude", "longitude", "elevation"]].drop_duplicates().to_numpy())
groups = build_groups(df, scaler, loc_scaler)
train_idx = build_index(groups, TRAIN_START, TRAIN_END)
val_idx = build_index(groups, VAL_START, VAL_END)
test_idx = build_index(groups, TEST_START, TEST_END)
print("Windows: train", len(train_idx), "validation", len(val_idx), "test", len(test_idx))


## 6. Load the saved LSTM or train it once

In [ ]:
if REUSE_MODEL:
    model = tf.keras.models.load_model(MODEL_DIR / "model.keras", compile=False)
    print("Loaded:", MODEL_DIR / "model.keras")
else:
    if not TRAIN_IF_MISSING and not FORCE_RETRAIN:
        raise FileNotFoundError("No complete saved model. Enable TRAIN_IF_MISSING or restore the artifacts.")
    if not (pd.Timestamp(TRAIN_END) < pd.Timestamp(VAL_START) <= pd.Timestamp(VAL_END) < pd.Timestamp(TEST_START) <= pd.Timestamp(TEST_END)):
        raise ValueError("Training, validation and test dates must be ordered and disjoint.")
    if min(len(train_idx), len(val_idx), len(test_idx)) == 0:
        raise ValueError("Training requires nonempty train, validation and test windows.")
    rng = np.random.default_rng(SEED)
    if MAX_TRAIN_WINDOWS is not None and len(train_idx) > MAX_TRAIN_WINDOWS:
        train_idx = train_idx[rng.choice(len(train_idx), MAX_TRAIN_WINDOWS, replace=False)]
    tf.keras.utils.set_random_seed(SEED)
    past = tf.keras.layers.Input(shape=(LOOKBACK, len(features)+5), name="past_weather_calendar_location")
    target = tf.keras.layers.Input(shape=(2,), name="target_day_calendar")
    z = tf.keras.layers.LSTM(96, return_sequences=True)(past)
    z = tf.keras.layers.LSTM(64)(z)
    z = tf.keras.layers.Concatenate()([z, target])
    z = tf.keras.layers.Dense(96, activation="relu")(z)
    output = tf.keras.layers.Dense(len(features), name="next_day_weather")(z)
    model = tf.keras.Model([past, target], output)
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss="mse", metrics=["mae"])

    def window_dataset(indices, shuffle):
        def generate():
            order = indices.copy()
            if shuffle:
                rng.shuffle(order)
            for offset in range(0, len(order), BATCH_SIZE):
                xx, cc, yy = input_batch(order[offset:offset+BATCH_SIZE], groups)
                yield (xx, cc), yy
        signature = ((tf.TensorSpec((None, LOOKBACK, len(features)+5), tf.float32),
                      tf.TensorSpec((None, 2), tf.float32)),
                     tf.TensorSpec((None, len(features)), tf.float32))
        dataset = tf.data.Dataset.from_generator(generate, output_signature=signature)
        options = tf.data.Options()
        options.threading.private_threadpool_size = 1
        dataset = dataset.apply(tf.data.experimental.assert_cardinality((len(indices)+BATCH_SIZE-1)//BATCH_SIZE))
        return dataset.with_options(options).prefetch(1)

    history = model.fit(window_dataset(train_idx, True), validation_data=window_dataset(val_idx, False),
                        epochs=EPOCHS, callbacks=[
                            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
                            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)], verbose=2)
    pd.DataFrame(history.history).to_csv(LOG_DIR / "training_history.csv", index=False)
    pd.DataFrame(history.history)[["loss", "val_loss"]].plot(figsize=(8, 4))
    plt.ylabel("Scaled MSE")
    plt.xlabel("Epoch")
    plt.title("Training and validation loss")
    plt.savefig(LOG_DIR / "training_loss.png", dpi=120, bbox_inches="tight")
    plt.show()
    model.save(MODEL_DIR / "model.keras")
    joblib.dump({"scaler": scaler, "loc_scaler": loc_scaler}, MODEL_DIR / "scalers.joblib")
    config = {"features": features, "lookback": LOOKBACK, "train_end": TRAIN_END,
              "validation": f"{VAL_START} to {VAL_END}", "test": f"{TEST_START} to {TEST_END}",
              "direction_encoding": "sin_cos", "training_input": str(DATA_PATH), "experiment_id": EXPERIMENT_ID}
    (MODEL_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
if model.input_shape != [(None, LOOKBACK, len(features)+5), (None, 2)] or model.output_shape != (None, len(features)):
    raise ValueError("Saved model shapes do not match the original weather feature/scaler contract.")
model.summary()


## 7. Compile batched inference and record the experiment

In [ ]:
@tf.function(input_signature=[tf.TensorSpec((None, LOOKBACK, len(features)+5), tf.float32),
                              tf.TensorSpec((None, 2), tf.float32)])
def forecast_batch(past_weather, target_calendar):
    return model([past_weather, target_calendar], training=False)

def forecast_call(inputs, training=False):
    return forecast_batch(tf.convert_to_tensor(inputs[0], dtype=tf.float32),
                          tf.convert_to_tensor(inputs[1], dtype=tf.float32))

experiment = {"experiment_id": EXPERIMENT_ID, "features": features, "lookback": LOOKBACK,
              "reused_model": REUSE_MODEL, "saved_model_config": config,
              "forecast_start_setting": FORECAST_START, "forecast_end_setting": FORECAST_END,
              "selected_prefecture": FORECAST_PREFECTURE, "selected_town": FORECAST_TOWN,
              "forecast_batch_size": FORECAST_BATCH_SIZE,
              "training_settings": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "seed": SEED,
                                    "max_train_windows": MAX_TRAIN_WINDOWS}}
(LOG_DIR / "experiment_config.json").write_text(json.dumps(experiment, indent=2), encoding="utf-8")


## 8. Metrics on heldout historical weather
MAE, RMSE and bias are reported in each attribute’s units. Wind direction uses circular angular error. A persistence baseline predicts the last observed value. One-step prediction uses observed history for every target; recursive backtesting feeds predictions back through the whole test period.

In [ ]:
def physical_rows(scaled_predictions):
    raw = _bound_weather(scaler.inverse_transform(np.asarray(scaled_predictions)), features)
    rows = []
    for values in raw:
        values = dict(zip(features, map(float, values)))
        if "wind_direction_sin" in values and "wind_direction_cos" in values:
            values["wind_direction_10m_dominant"] = float(np.degrees(np.arctan2(values.pop("wind_direction_sin"), values.pop("wind_direction_cos"))) % 360)
        rows.append(values)
    return pd.DataFrame(rows)

def metric_tables(predicted, actual, previous):
    join = ["prefecture", "town", "date"]
    aligned = predicted.merge(actual, on=join, suffixes=("_predicted", "_actual"), validate="one_to_one")
    if len(aligned) != len(predicted):
        raise ValueError("Some forecast dates lack matching observed weather; metrics cannot silently drop targets.")
    baseline_keys = join if "date" in previous else ["prefecture", "town"]
    aligned = aligned.merge(previous, on=baseline_keys, validate="one_to_one" if "date" in previous else "many_to_one")
    physical_features = [c for c in features if c not in {"wind_direction_sin", "wind_direction_cos"}]
    if "wind_direction_sin" in features:
        physical_features += [ANGLE]
    def summarize(frame, horizon=None):
        result=[]
        for feature in physical_features:
            observed=frame[feature + "_actual"].to_numpy()
            estimate=frame[feature + "_predicted"].to_numpy()
            baseline=frame[feature + "_baseline"].to_numpy()
            error=estimate-observed; baseline_error=baseline-observed
            if feature == ANGLE:
                error=(error+180)%360-180; baseline_error=(baseline_error+180)%360-180
            mae=float(np.abs(error).mean()); baseline_mae=float(np.abs(baseline_error).mean())
            result.append({"feature":feature, "MAE":mae, "RMSE":float(np.sqrt(np.mean(error**2))),
                           "bias":float(np.mean(error)), "persistence_MAE":baseline_mae,
                           "MAE_skill_vs_persistence": None if baseline_mae == 0 else 1-mae/baseline_mae,
                           "count":len(error), "horizon_days":horizon})
        return pd.DataFrame(result)
    metrics=summarize(aligned)
    horizons=pd.concat([summarize(aligned[aligned.horizon_day <= h], h) for h in sorted(set([1,7,14,31,int(aligned.horizon_day.max())]))
                       if "horizon_day" in aligned and (aligned.horizon_day<=h).any() and h<=aligned.horizon_day.max()],ignore_index=True) if "horizon_day" in aligned else None
    return metrics, horizons, aligned

if RUN_EVALUATION:
    trained_end = pd.Timestamp(config["train_end"])
    saved_validation_end = pd.Timestamp(config["validation"].split(" to ")[-1])
    if pd.Timestamp(TEST_START) <= max(trained_end, saved_validation_end):
        raise ValueError("The selected test period overlaps saved model training/validation; choose a genuine heldout period.")
    if len(test_idx)==0:
        raise ValueError("No observed weather for the configured heldout test dates.")
    chunks=[]
    for offset in range(0,len(test_idx),FORECAST_BATCH_SIZE):
        pairs=test_idx[offset:offset+FORECAST_BATCH_SIZE]
        xx,cc,_=input_batch(pairs,groups)
        values=physical_rows(forecast_call([xx,cc]))
        values["prefecture"]=[groups[int(g)]["prefecture"] for g,t in pairs]
        values["town"]=[groups[int(g)]["town"] for g,t in pairs]
        values["date"]=[groups[int(g)]["dates"][int(t)] for g,t in pairs]
        chunks.append(values)
    one_step=pd.concat(chunks,ignore_index=True)
    recursive=recursive_forecast(df,forecast_call,scaler,loc_scaler,features,LOOKBACK,TEST_START,TEST_END,
                                 prefecture=FORECAST_PREFECTURE,town=FORECAST_TOWN,
                                 batch_size=FORECAST_BATCH_SIZE,progress_every=None)
    recursive["date"]=pd.to_datetime(recursive["date"])
    recursive["horizon_day"]=(recursive.date-pd.Timestamp(TEST_START)).dt.days+1
    selected_keys=recursive[["prefecture","town"]].drop_duplicates()
    one_step=one_step.merge(selected_keys,on=["prefecture","town"],validate="many_to_one")
    actual=df[df.date.between(TEST_START,TEST_END)].merge(selected_keys,on=["prefecture","town"],validate="many_to_one")
    if ANGLE not in actual:
        actual[ANGLE]=np.degrees(np.arctan2(actual.wind_direction_sin,actual.wind_direction_cos))%360
    previous=df[df.date<pd.Timestamp(TEST_START)].sort_values("date").groupby(["prefecture","town"],as_index=False).tail(1)
    metric_names=[c for c in one_step if c not in {"prefecture","town","date"}]
    previous=previous[["prefecture","town"]+metric_names].rename(columns={c:c+"_baseline" for c in metric_names})
    one_step_baseline=df[["prefecture","town","date"]+metric_names].copy()
    one_step_baseline["date"] += pd.Timedelta(days=1)
    one_step_baseline=one_step_baseline.rename(columns={c:c+"_baseline" for c in metric_names})
    one_step_metrics,_,one_step_comparison=metric_tables(one_step,actual,one_step_baseline)
    recursive_metrics,horizon_metrics,recursive_comparison=metric_tables(recursive,actual,previous)
    one_step_metrics.to_csv(LOG_DIR/"one_step_metrics.csv",index=False)
    recursive_metrics.to_csv(LOG_DIR/"recursive_test_metrics.csv",index=False)
    if horizon_metrics is not None:
        horizon_metrics.to_csv(LOG_DIR/"recursive_horizon_metrics.csv",index=False)
    one_step_comparison.to_csv(LOG_DIR/"one_step_historical_predictions.csv",index=False,date_format="%Y-%m-%d")
    recursive_comparison.to_csv(LOG_DIR/"recursive_historical_predictions.csv",index=False,date_format="%Y-%m-%d")
    print("One-step heldout metrics:")
    display(one_step_metrics.sort_values("RMSE",ascending=False))
    print("Recursive heldout metrics:")
    display(recursive_metrics.sort_values("RMSE",ascending=False))
else:
    print("Historical evaluation skipped by RUN_EVALUATION=False.")


## 9. View historical comparisons

In [ ]:
if RUN_EVALUATION:
    for feature in ["temperature_2m_mean","relative_humidity_2m_mean","pressure_msl_mean","wind_speed_10m_max","rain_sum"]:
        if feature+"_predicted" not in recursive_comparison:
            continue
        daily=recursive_comparison.groupby("date")[[feature+"_actual",feature+"_predicted"]].mean()
        ax=daily.plot(figsize=(9,4),title=f"Recursive historical comparison — {feature} (town average)")
        ax.set_ylabel(feature)
        ax.legend(["Actual","Recursive forecast"])
        plt.tight_layout()
        plt.savefig(LOG_DIR/f"recursive_comparison_{feature}.png",dpi=120,bbox_inches="tight")
        plt.show()


## 10. Forecast one complete year
Towns are predicted together in batches for each date. The output must contain every requested date for every selected town. Only this future CSV goes in `Forecast`; its coverage report goes in `experiment_logs`.

In [ ]:
if RUN_FORECAST:
    selected=df
    if FORECAST_PREFECTURE is not None:
        selected=selected[selected.prefecture==FORECAST_PREFECTURE]
    if FORECAST_TOWN is not None:
        selected=selected[selected.town==FORECAST_TOWN]
    if selected.empty:
        raise ValueError("No towns match the selected filters.")
    start=pd.Timestamp(FORECAST_START).normalize() if FORECAST_START is not None else selected.date.max()+pd.Timedelta(days=1)
    end=pd.Timestamp(FORECAST_END).normalize() if FORECAST_END is not None else start+pd.DateOffset(years=1)-pd.Timedelta(days=1)
    print("Requested forecast:",start.date(),"through",end.date(),"| Days:",(end-start).days+1)
    future_df=recursive_forecast(df,forecast_call,scaler,loc_scaler,features,LOOKBACK,start,end,
                                prefecture=FORECAST_PREFECTURE,town=FORECAST_TOWN,
                                batch_size=FORECAST_BATCH_SIZE,progress_every=PROGRESS_EVERY)
    future_df["date"]=pd.to_datetime(future_df["date"])
    # Every input needed by the weather-only typhoon classifier must be available.
    classifier_weather=['temperature_2m_mean','temperature_2m_max','temperature_2m_min',
        'relative_humidity_2m_mean','relative_humidity_2m_max','relative_humidity_2m_min',
        'dew_point_2m_mean','dew_point_2m_max','dew_point_2m_min','precipitation_sum','rain_sum','precipitation_hours',
        'cloud_cover_mean','cloud_cover_max','cloud_cover_min','pressure_msl_mean','pressure_msl_max','pressure_msl_min',
        'surface_pressure_mean','surface_pressure_max','surface_pressure_min','wind_speed_10m_mean','wind_speed_10m_max',
        'wind_speed_10m_min','wind_gusts_10m_mean','wind_gusts_10m_max','wind_gusts_10m_min',
        'wind_direction_10m_dominant','shortwave_radiation_sum']
    missing=set(classifier_weather)-set(future_df)
    if missing or not np.isfinite(future_df[classifier_weather].to_numpy()).all():
        raise ValueError(f"Weather model cannot supply all classifier weather fields: {sorted(missing)}.")
    path=FORECAST_DIR/"future_daily_weather_attributes.csv"
    future_df.to_csv(path,index=False,encoding="utf-8-sig",date_format="%Y-%m-%d",float_format="%.2f")
    towns=future_df.groupby(["prefecture","town"]).ngroups
    coverage={"start":str(start.date()),"end":str(end.date()),"days_per_town":(end-start).days+1,
              "towns":towns,"expected_rows":towns*((end-start).days+1),"actual_rows":len(future_df),
              "all_dates_verified":True,"future_accuracy":None,
              "note":"Future accuracy requires observed weather; see heldout historical metrics."}
    (LOG_DIR/"forecast_coverage.json").write_text(json.dumps(coverage,indent=2),encoding="utf-8")
    print("Saved future weather:",path)
    print(f"Verified {towns} towns × {coverage['days_per_town']} days = {len(future_df):,} rows")
    display(future_df.head())
else:
    print("Forecast skipped by RUN_FORECAST=False.")


## 11. Preview the annual weather curve

In [ ]:
if RUN_FORECAST:
    first=future_df[["prefecture","town"]].drop_duplicates().iloc[0]
    preview=future_df[(future_df.prefecture==first.prefecture)&(future_df.town==first.town)].set_index("date")
    for feature in ["temperature_2m_mean","pressure_msl_mean","rain_sum"]:
        if feature in preview:
            preview[feature].plot(figsize=(10,3),title=f"{first.prefecture}/{first.town} — {feature}")
            plt.ylabel(feature)
            plt.tight_layout()
            plt.savefig(LOG_DIR/f"annual_forecast_{feature}.png",dpi=120,bbox_inches="tight")
            plt.show()
